In [1]:
import pyodbc
import pandas as pd

# Connect to SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Connected successfully")

Connected successfully


In [ ]:
# View 1: Cohort Analysis
cohort_sql = """
CREATE VIEW vw_cohort_analysis AS
WITH tbl_acquisition AS (
    SELECT 
        CustomerID,
        DATEFROMPARTS(YEAR(MIN(InvoiceDate)), MONTH(MIN(InvoiceDate)), 1) AS CohortMonth
    FROM retail_sales
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
),
tbl_activity AS (
    SELECT
        r.CustomerID, 
        DATEFROMPARTS(YEAR(r.InvoiceDate), MONTH(r.InvoiceDate), 1) AS ActivityMonth,
        a.CohortMonth
    FROM retail_sales AS r
    JOIN tbl_acquisition AS a ON r.CustomerID = a.CustomerID
),
tbl_cohort AS (
    SELECT
        CohortMonth,
        ActivityMonth,
        DATEDIFF(MONTH, CohortMonth, ActivityMonth) AS MonthNumber,
        COUNT(DISTINCT CustomerID) AS ActiveCustomers
    FROM tbl_activity
    GROUP BY CohortMonth, ActivityMonth, DATEDIFF(MONTH, CohortMonth, ActivityMonth)
)
SELECT CohortMonth, MonthNumber, ActiveCustomers
FROM tbl_cohort
"""

In [ ]:
# View 2: A/B Test
ab_test_sql = """
CREATE VIEW vw_ab_test_summary AS
SELECT
    TestGroup,
    COUNT(CustomerID) AS NumberOfCustomers,
    ROUND(AVG(TotalRevenue), 2) AS AvgCustomerRevenue,
    ROUND(AVG(AvgOrderValue), 2) AS AvgOrderValue,
    ROUND(AVG(TotalOrders), 2) AS AvgOrders
FROM vw_ab_test
GROUP BY TestGroup
"""

In [ ]:
# View 3: A/B Test Summary  
ab_summary_sql = """
CREATE VIEW vw_ab_test_summary AS
SELECT
    TestGroup,
    COUNT(CustomerID) AS NumberOfCustomers,
    ROUND(AVG(TotalRevenue), 2) AS AvgCustomerRevenue,
    ROUND(AVG(AvgOrderValue), 2) AS AvgOrderValue,
    ROUND(AVG(TotalOrders), 2) AS AvgOrders
FROM vw_ab_test
GROUP BY TestGroup
"""

In [ ]:
print("SQL views documented")

In [2]:
import scipy.stats as stats
import numpy as np

# Pull individual customer revenue for each group
ab_data = pd.read_sql("""
    SELECT TestGroup, AvgOrderValue
    FROM vw_ab_test
    WHERE AvgOrderValue IS NOT NULL
""", conn)

C:\Users\rashi\AppData\Local\Temp\ipykernel_61464\2786501628.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ab_data = pd.read_sql("""


In [4]:
# Split into two groups
control = ab_data[ab_data['TestGroup'] == 'Control']['AvgOrderValue']
treatment = ab_data[ab_data['TestGroup'] == 'Treatment']['AvgOrderValue']

print(f"Control group: {len(control)} customers, mean AOV: £{control.mean():.2f}")
print(f"Treatment group: {len(treatment)} customers, mean AOV: £{treatment.mean():.2f}")
print(f"Difference: £{treatment.mean() - control.mean():.2f}")

Control group: 2172 customers, mean AOV: £98.46
Treatment group: 2166 customers, mean AOV: £38.16
Difference: £-60.30


In [5]:
# Run independent samples t-test
t_stat, p_value = stats.ttest_ind(control, treatment)

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Interpret the result
alpha = 0.05
if p_value < alpha:
    print(f"\nResult: STATISTICALLY SIGNIFICANT (p={p_value:.4f} < {alpha})")
    print("The difference in average order value between groups is unlikely to be due to chance.")
else:
    print(f"\nResult: NOT STATISTICALLY SIGNIFICANT (p={p_value:.4f} >= {alpha})")
    print("The difference in average order value between groups could be due to chance.")


t-statistic: 1.3529
p-value: 0.1762

Result: NOT STATISTICALLY SIGNIFICANT (p=0.1762 >= 0.05)
The difference in average order value between groups could be due to chance.
